# **종합실습_나만의 챗봇 만들기**

* 예비 에이블러들을 위한 QA 챗봇 모델 만들기
    * Vector DB에 데이터 추가하기
    * Retriever, memory, LLM를 연결하기

## **1.환경준비**

### (1) 구글 드라이브

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### (2) 라이브러리

In [1]:
!pip install langchain langchain-community openai chromadb tiktoken -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 884.7 kB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.7 MB/s eta 0:00:0

In [3]:
import pandas as pd
import numpy as np
import os
import sqlite3
from datetime import datetime

import openai

from langchain.chat_models import ChatOpenAI
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma, FAISS
from langchain.chains import RetrievalQA, ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory, ConversationSummaryBufferMemory

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [4]:
def load_api_keys(filepath="api_key.txt"):
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if line and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

path = '/content/drive/MyDrive/langchain/'

# API 키 로드 및 환경변수 설정
load_api_keys(path + 'api_key.txt')

In [5]:
print(os.environ['OPENAI_API_KEY'][:40])

sk-6HnpTor5FDdHd3WXGaRUshfzSdbIiNeoKz1V4


## **2.Vector DB 만들기**

* 데이터 로딩
    * 제공한 csv 파일의 구조를 그대로 이용
    * 에이블스쿨 홈페이지 FAQ 데이터 수집(https://aivle.kt.co.kr/home/brd/faq/main?mcd=MC00000056)
        * 질문들을 csv 형태로 저장
    * csv 로더로 로딩하기

In [6]:
from langchain.document_loaders import CSVLoader

# CSV 파일 로드
csv_path = "file.csv"
csv_loader = CSVLoader(file_path= path + csv_path)

# 문서 로드 실행
documents_csv = csv_loader.load()

# 첫 번째 행 출력
print(f"총 {len(documents_csv)} 개의 행이 로드됨")
print(documents_csv[0].page_content)
print(documents_csv[0].metadata)

총 327 개의 행이 로드됨
﻿약어: 2D
영문: Two dimensional space Drawing
국문: 2차원 설계
구분: 공식 약어
내용기술: ● 정의(개념)
2D(Two dimensional space Drawing)는 가로(x축)와 세로(y축) 두 개의 축만 사용하여 평면상에 물체의 형태나 구조를 표현하는 설계 방식을 의미한다. 입체적인 깊이(z축)의 개념이 없이, 평면적인 정보만을 담고 있다.

● 사용 용도 및 활용 측면
2D 설계는 대상의 높이와 너비 등 2차원 정보만을 표현하며, 깊이감이나 입체감은 나타내지 않는다. 평면 도면은 시각적으로 이해하기 쉽고, 간단한 형태를 빠르게 표현할 수 있어 설계 초기나 기술 커뮤니케이션에 유리하며, 3D 설계에 비해 필요한 소프트웨어나 기술적 복잡성이 낮아 접근성이 뛰어나다. 2D는 제품 설계, 기계 설계, 전자 회로 설계, 패션 디자인, 그래픽 디자인 등 다양한 분야에서 기본적인 도면 작성 및 정보 전달에 활용된다.

● 우리 회사에서 사용하는 측면
ANAM에서는 2D 설계를 주로 연구소 각 부서에서 기본 도면 작성 및 기술 정보 전달 수단으로 활용한다. 제품의 부품, 전자 회로 등의 평면도나 입면도는 제품의 형태와 치수를 정확히 표현하여 설계 의도를 명확히 전달하며, 이는 생산, 시공, 유지보수 단계에서 기초 자료로 활용된다. 또한, 외부 협력업체와의 기술 협의 시에도 2D 도면은 주요 커뮤니케이션 도구로 사용되며, 부품 공급업체 및 시공업체 등과의 공동 작업에 있어 기술 정보를 효율적으로 교환한다. ANAM의 기구팀은 고객에게는 제품의 디자인, 공간 구성 등을 2D 도면으로 제시함으로써 이해도를 높이고, 최종 의사 결정 과정에서의 신뢰성과 설득력을 높이는 데 기여한다.
사용예시: "IQC 검사를 할 수 있도록 치수/공차가 표기된 2D 도면 발행합니다."
출처: 기구팀
: 
{'source': '/content/drive/MyDrive/langchain/file.csv', 'row': 0}


* 벡터 데이터베이스
    * Embedding 모델 : text-embedding-ada-002
    * DB 경로 : ./db



In [7]:
embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")
database = Chroma.from_documents(documents_csv, embeddings, persist_directory="./db")

## **3.RAG 파이프라인**

* 모델 : ConversationalRetrievalChain
    * LLM 모델 : gpt-4o-mini, temperater
    * retriever : 벡터DB
        * 유사도 높은 문서 3개 가져오도록 설정
    * 요약 memory
* ChatPromptTemplate로 역할 지정

In [8]:
# (1) 리트리버(Retriever) 생성
retriever = database.as_retriever(search_kwargs={"k": 2})

# (2) GPT-4o mini 모델 설정
llm = ChatOpenAI(model_name="gpt-4o-mini")

# (3) 메모리 추가 (대화 문맥 유지)
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True, output_key="answer")

In [22]:
from langchain.prompts import ChatPromptTemplate

# 프롬프트 템플릿
prompt = ChatPromptTemplate.from_messages([
    ("system", '''
# 지침
- 너는 검색 결과 대로만 답해. 무엇을 추가하지 말고 검색된 내용을 그대로 답변해.
- 만약 지식에 없다면, 등록된 용어집에 없는 내용이라고 답해.

# 답변 형식
[약어]
[영문]
[국문]
[구분]
[내용기술]
[사용예시]
[출처]
'''),
    ("human", "질문: {question}\n\n관련 문서:\n{context}")
])

# 체인 구성
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    combine_docs_chain_kwargs={"prompt": prompt},
    return_source_documents=True,
)


* 모델 사용

In [23]:
# 질문
query = "3정 5S"   # 질문할 문장

# 답변
result = qa_chain(query)
answer = result["answer"]
print(answer)

[약어] 3정 5S  
[영문] 3정 / 5S  
[국문] 3정(정품, 정량, 정위치), 5S(정리, 정돈, 청소, 청결, 습관화)  
[구분] 공식 약어  
[내용기술] ●정의(개념) 3정(정위치, 정량, 정품), 5S(정리, 정돈, 청소, 청결, 습관화)는 사업장 관리의 기본이 되는 요소다. 작업 환경을 제대로 정리, 정돈하는 것만으로 큰 낭비나 손실을 줄이고 안전 목표를 실현하는 것이다.  

●사용 용도 및 활용 측면  
1) 정품 : 보관해야 할 품목을 정하고, 보관하는 방법을 결정하여 물건의 품명을 표시하는 것이다.  
2) 정량 : 보관 물품의 사용 상태를 파악하여 적정한 보관 수량(최대 / 최소 재고)을 표시하는 것이다.  
3) 정위치 : 보관 위치를 결정하여 주소를 표시하여 정해진 위치를 명확히 하는 것이다.  
4) 정리 : 필요한 것과 불필요한 것을 구분하고 불필요한 것을 버리는 것이다.  
5) 정돈 : 누구나 쉽게 찾을 수 있고 언제나 사용할 수 있게 하는 것이다.  
6) 청소 : 먼지, 더러운 이물질 등을 없애고 깨끗하게 하는 것이다.  
7) 청결동 : 정리, 정돈, 청소를 반복하여 항상 깨끗한 상태를 유지하는 것이다.  
8) 습관화 : 정리, 정돈, 청소, 청결을 습관처럼 유지하는 것이다.  

●우리 회사에서 사용되는 측면  
3정 5S는 단순히 정리와 청소에 국한되지 않고, 당사의 운영 효율을 극대화하며 안전한 작업 환경을 조성하는 핵심 관리 기법이다. 이를 지속적으로 실천하면 당사의 생산성 향상, 품질 개선, 비용 절감, 그리고 근로자 만족도를 동시에 달성할 수 있게 된다.  
[사용예시] "3정 5S를 관리를 통해 큰 낭비나 손실을 줄이고 안전 목표를 실현한다."  
[출처] VN(PE)
